# ToolInstance Persistence

Every tool in `bio_programming_tools` runs in an **isolated virtual environment** via
`ToolInstance`. This keeps heavy dependencies (PyTorch, ESM, etc.) out of the main
environment while providing a clean interface for tool execution.

For ESMFold, loading the model takes ~30 seconds while inference on a short sequence takes <1 second. By default, each `run_*` call spins up a fresh subprocess (safe, no leaks), but for batch workloads
this means reloading the model every call.

`ToolInstance` provides opt-in persistence to solve this:

| Method | Caches? | Cleanup | Best for |
|---|---|---|---|
| Default (one-shot) | No | Automatic | Single calls, safety-first |
| `ToolInstance.persist()` | Yes, auto | Automatic on exit | Batch jobs, optimization loops |
| `ToolInstance.persist_tool()` | Yes, only the named tool | Automatic on exit | Multi-GPU, multiple instances |
| `ToolInstance.get()` | Yes, until close | Manual | Long-running sessions |

**Device handling:** Device is set via `config.device` (a `BaseConfig` field, default
`"cpu"`). GPU tool configs override this to `"cuda"`. The device flows through
`input_dict` to set `CUDA_VISIBLE_DEVICES` in the subprocess.

**Auto-restart on config changes:** Persistent workers automatically restart when any
config field marked `reload_on_change=True` changes between calls. This includes
`device`, `model_checkpoint`, `model_name`, and other load-time parameters — no need
to manually close and reopen when switching models or devices.

**Timeout:** Every tool call has a configurable timeout (default 600s / 10 minutes).
Set `config.timeout` to override. On timeout, the subprocess is killed and a
`TimeoutError` is raised — works for both one-shot and persistent modes.

This notebook demonstrates each mode using ESMFold as the example tool.

In [1]:
from bio_programming_tools.tools.structure_prediction.esmfold import (
    run_esmfold, ESMFoldInput, ESMFoldConfig,
)
from bio_programming_tools.utils.tool_instance import ToolInstance, _instances
from time import time

---
## 1. Default Behavior (One-Shot)

Every `run_*` function uses one-shot execution by default. Each call spins up an
ephemeral subprocess, loads the model, runs inference, and exits. No leaked processes,
no GPU memory retained.

In [2]:
# Two consecutive calls — each loads the model from scratch
start = time()
output1 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
t1 = time() - start
print(f"First call:  {t1:.1f}s (model load + inference)")

start2 = time()
output2 = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda"),
)
t2 = time() - start2
print(f"Second call: {t2:.1f}s (model load + inference again)")
print(f"Total:       {t1 + t2:.1f}s")

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Folding structures (ESMFold): 100%|██████████| 1/1 [00:38<00:00, 38.48s/structure]


First call:  38.5s (model load + inference)


Folding structures (ESMFold): 100%|██████████| 1/1 [00:38<00:00, 38.87s/structure]

Second call: 38.9s (model load + inference again)
Total:       77.4s


In [3]:
# No persistent instance was created — the cache is empty
print(f"Cached instances: {dict(_instances)}")
assert len(_instances) == 0, "One-shot calls should not leave anything in the cache"

Cached instances: {}


---
## 2. `persist()` Context Manager (Recommended)

The simplest way to use persistence. Like `torch.inference_mode()`, just wrap your
code — any tool called inside is auto-cached on first use, and everything is cleaned
up on exit. No tool names needed upfront:

In [4]:
sequences = ["MKTLLILAVVAAALA", "MKTLLILAVVAAALA"]

start = time()
with ToolInstance.persist():
    for i, seq in enumerate(sequences):
        call_start = time()
        output = run_esmfold(
            ESMFoldInput(complexes=[seq]),
            ESMFoldConfig(device="cuda"),
        )
        elapsed = time() - call_start
        print(f"Call {i+1}: {elapsed:.1f}s {'(model load + inference)' if i == 0 else '(inference only)'}")

total = time() - start
print(f"Total: {total:.1f}s")

# Block exited — all auto-cached workers killed, GPU memory freed
print(f"\nAfter block — cached: {list(_instances.keys())}")
assert len(_instances) == 0

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Folding structures (ESMFold): 100%|██████████| 1/1 [00:39<00:00, 39.41s/structure]


Call 1: 39.4s (model load + inference)


Folding structures (ESMFold): 100%|██████████| 1/1 [00:00<00:00,  1.23structure/s]


Call 2: 0.9s (inference only)
Total: 40.9s

After block — cached: []


---
## 3. `persist_tool()` — Tool-Specific Persistence

When you need to name a specific tool or manage multiple instances (e.g., multi-GPU),
use `persist_tool()` with an explicit tool name. For a single tool, the simple form
works just like `persist()` but scoped to one tool:

In [5]:
sequences = ["MKTLLILAVVAAALA", "MKTLLILAVVAAALA"]

start = time()
with ToolInstance.persist_tool("esmfold"): # when you only have one instance of a tool, you don't need an instance name
    for i, seq in enumerate(sequences):
        call_start = time()
        output = run_esmfold(
            ESMFoldInput(complexes=[seq]),
            ESMFoldConfig(device="cuda"), #<-- no need to pass instance here, it will be found automatically
        )
        elapsed = time() - call_start
        print(f"Call {i+1}: {elapsed:.1f}s {'(model load + inference)' if i == 0 else '(inference only)'}")

total = time() - start
print(f"Total: {total:.1f}s")

# Block exited — worker killed, GPU memory freed
print(f"\nAfter block — cached: {list(_instances.keys())}")
assert len(_instances) == 0

Folding structures (ESMFold): 100%|██████████| 1/1 [00:37<00:00, 37.34s/structure]


Call 1: 37.4s (model load + inference)


Folding structures (ESMFold): 100%|██████████| 1/1 [00:00<00:00,  1.15structure/s]


Call 2: 0.9s (inference only)
Total: 38.9s

After block — cached: []


---
### Explicit Instance Passing (Multi-Instance / Multi-GPU)

When you need multiple instances of the same tool running simultaneously, use named
instances and pass `instance=` to route calls to a specific worker. This is useful
for multi-GPU setups or running different model checkpoints side by side.

```python
# Two named instances of the same tool — each gets its own worker process
with ToolInstance.persist_tool("esmfold", instance_name="worker_a") as inst_a: # you can get the instance object
    with ToolInstance.persist_tool("esmfold", instance_name="worker_b"): # or you can just use the name
        print(f"Cached instances: {list(_instances.keys())}")

        # Route each call to a specific worker (must specify when there are multiple instances of the same tool)
        out_a = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:0"),
            instance=inst_a, # you can pass the instance object
        )

        out_b = run_esmfold(
            ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
            ESMFoldConfig(device="cuda:1"),
            instance="worker_b", # or you can pass the instance name
        )
```

---
## 4. `get()` / `shutdown()` — Manual Lifecycle

For long-running sessions (e.g. interactive notebooks) where you want to keep a model
warm across multiple cells. You control when the worker is created and destroyed.

Call `tool.shutdown()` when done — this stops the worker **and** evicts the instance
from the cache.

In [6]:
# Create a persistent instance — stays cached until explicitly shut down
tool = ToolInstance.get("esmfold")
print(f"Cached: {list(_instances.keys())}")

# No instance= needed — dispatch auto-finds the cached instance
for seq in sequences:
    output = run_esmfold(
        ESMFoldInput(complexes=[seq]),
        ESMFoldConfig(device="cuda"),
    )
    print(f"  pLDDT={output.structures[0].avg_plddt:.1f}")

# Clean up when done — stops worker and evicts from cache
tool.shutdown()
print(f"After shutdown: {list(_instances.keys())}")
assert len(_instances) == 0

Cached: ['esmfold']


Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

Folding structures (ESMFold): 100%|██████████| 1/1 [00:37<00:00, 37.54s/structure]


  pLDDT=0.9


Folding structures (ESMFold): 100%|██████████| 1/1 [00:00<00:00,  1.10structure/s]


  pLDDT=0.9
After shutdown: []


In [7]:
# You can also shut down instances by name at the class level,
# without needing a reference to the instance object
tool = ToolInstance.get("esmfold")
print(f"Cached: {list(_instances.keys())}")

ToolInstance.shutdown_instance("esmfold")
print(f"After shutdown_instance: {list(_instances.keys())}")
assert len(_instances) == 0

# If the tool function is called after this point it will be a one-shot call

Cached: ['esmfold']
After shutdown_instance: []


---
## 5. Auto-Restart on Config Changes

Persistent workers automatically restart when any config field marked with
`reload_on_change=True` changes between calls. This includes `device`,
`model_checkpoint`, `model_name`, and other load-time parameters.

There is no device parameter on `persistent()` or `get()` — the worker detects
config mismatches from the values in `input_dict` and restarts transparently.

`device` is a field on `BaseConfig`, so every tool config inherits it. GPU tools
default to `"cuda"` via their shared base config; CPU tools default to `"cpu"`.
Other `reload_on_change` fields (like `model_checkpoint`) are defined per tool.

---
## 6. Timeout

Every tool call enforces a configurable timeout (default: 600 seconds / 10 minutes).
If a call exceeds the timeout, the subprocess is killed and a `TimeoutError` is raised.
This works for both one-shot and persistent modes.

Override the default via `config.timeout`:

```python
# Set a 120-second timeout
output = run_esmfold(
    ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
    ESMFoldConfig(device="cuda", timeout=120),
)
```

For persistent workers, the timeout applies per-call — the worker itself stays alive
for future calls even if one call times out.